In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown
from huggingface_hub import InferenceClient

In [ ]:
load_dotenv()
router=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
load_dotenv()
ollama=OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
hugging=InferenceClient(api_key='')
qwen='Qwen2.5-Coder-14B-Instruct'
deepseek="deepseek-ai/DeepSeek-V3-0324"
llama="llama3.2"

In [31]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HUGGING_FACE_API"]
)

models = client.models.list()

for model in models.data:
    print(model.id)

Qwen/Qwen3.8-27B
zai-org/GLM-5.3-Flash
deepseek-ai/DeepSeek-V4-Flash-0731
moonshotai/Kimi-K3
deepseek-ai/DeepSeek-V4-Pro-0813
meta-models/Muse-Glimmer-30B
ibm-granite/granite-4.2-30b
Qwen/Qwen3.8-2.4T-A95B
ibm-granite/granite-4.2-8b
ibm-granite/granite-4.2-3b
zai-org/GLM-5.2
zai-org/GLM-5.3-Flash-BF16
google/gemma-4-31B-it
prism-ml/Ternary-Bonsai-27B-gguf
meta-llama/Llama-3.1-8B-Instruct
openai/gpt-oss-20b
google/gemma-4-26B-A4B-it
deepseek-ai/DeepSeek-V4-Pro
inclusionAI/Ling-3.0-flash
Qwen/Qwen3.6-35B-A3B
tencent/Hy3
Qwen/Qwen3.5-9B
deepseek-ai/DeepSeek-V4-Flash
MiniMaxAI/MiniMax-M3
thinkingmachines/Inkling
thinkingmachines/Inkling-Small
Qwen/Qwen3-8B
Qwen/Qwen3-4B-Instruct-2507
openai/gpt-oss-120b
nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16
Qwen/Qwen3-Coder-Next
moonshotai/Kimi-K2.7-Code
Qwen/Qwen3-Coder-30B-A3B-Instruct
XiaomiMiMo/MiMo-V2.5
zai-org/GLM-5.2-FP8
meta-llama/Llama-3.3-70B-Instruct
Qwen/Qwen3.5-35B-A3B
Qwen/Qwen3.5-27B
Qwen/Qwen2.5-Coder-7B-Instruct
Qwen/Qwen3.6-27

In [3]:
system_prompt = """
Your task is to convert Python code into high performance rust code.
Respond only with rust code. Do not provide any explanation other than occasional comments.
The rust response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to rust with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called main.rust and then compiled and executed.
Respond only with rust code.
Python code to port:

```python
{python}
```
"""

In [4]:
def message_for(python):
    return [
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt_for(python)}
    ]

In [5]:
def save(rust):
    with open("main.rs","w",)as f:
        f.write(rust)

In [ ]:
def port(model,python):
    print("1. Function started")
    if model=='deepseek':
        response=hugging.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3-0324",
            messages=message_for(python),
            max_tokens=50000
        )
    elif model=='qwen':
        print("1. Function started")
        response=hugging.chat.completions.create(
                    model='Qwen/Qwen3.5-9B',
                    messages=message_for(python),
                    max_tokens=50000
        )
    else:
        response=ollama.chat.completions.create(
            model="llama3.2",
            messages=message_for(python)
        )
    code=response.choices[0].message.content
    code=code.replace('```cpp','').replace('```rust','').replace('```','')
    return code

1. Function started
1. Function started


In [9]:
def run_python(code):
    globals={"__builtins__":__builtins__}
    exec(code,globals)

In [10]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [11]:
run_python(python_hard)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 86.360188 seconds


In [21]:
port(deepseek,python_hard)

In [24]:
import gradio as gr

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python=gr.Textbox(label="python",lines=30)
        ru=gr.Textbox(label="rust",lines=30)
    with gr.Row():
        models=gr.Dropdown(['deepseek','qwen','lamma'],label="select a model",value='qwen')
        convert=gr.Button("convert")
    convert.click(port,inputs=[models,python],outputs=[ru])

ui.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


1. Function started
1. Function started
1. Function started
1. Function started
